In [ ]:
import os
os.chdir(path_to_wd)

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

import torch
print(torch.cuda.is_available()) 
import scvi
import anndata as ad
import scipy
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import os
import sys
import pycats
from pandas.api.types import CategoricalDtype

scvi.settings.seed = 1234
plt.rcParams['font.family']=['Dejavu serif']
plt.rcParams['figure.dpi'] = 100
plt.rcParams['pdf.fonttype'] = 'truetype'

In [ ]:
adata = sc.read("Reproducibility/Data/DOGMA/UC_DOGMA_RNA_ATAC_ADT_after_QC.h5ad")
adt_panel_path = "Reproducibility/Data/DOGMA/ADT_panel_list_for_MultiVI.txt"

In [ ]:
def prepare_and_train_multivi(
    adata,
    lineage="Global",
    adt_panel_path=adt_panel_path,
    rna_min_cells=100,
    atac_min_cell_frac=0.01,
    n_top_genes=3000,
    batch_key_in_obs="sample",
    multivi_batch_key="batch_id",
    max_epochs=1000,
    save_path="path_to_save"
):
    # ---- Select subset depending on lineage ----
    if lineage == "Global":
        adata_use = adata.copy()
    elif lineage == "Malignant":
        adata_use = adata[(adata.obs["lineage"] == "Epithelial") & (~adata.obs["celltype"].isin(["Normal"]))].copy()
    else:
        adata_use = adata[adata.obs["lineage"] == lineage].copy()

    # ---- 0) Split modalities ----
    adata_RNA  = adata_use[:, adata_use.var["modality"] == "Gene Expression"].copy()
    adata_ATAC = adata_use[:, adata_use.var["modality"] == "Peaks"].copy()

    # ---- 1) RNA gene filtering ----
    if "black_list" in adata_RNA.var.columns:
        keep_mask = adata_RNA.var["black_list"].astype(str).str.lower().eq("keep")
        adata_RNA = adata_RNA[:, keep_mask].copy()

    sc.pp.filter_genes(adata_RNA, min_cells=rna_min_cells)

    if lineage == "Global":
        sc.pp.highly_variable_genes(
            adata_RNA,
            n_top_genes=n_top_genes,
            flavor="seurat_v3",
            batch_key=batch_key_in_obs,
            subset=True
        )
    else:
        sc.pp.highly_variable_genes(
            adata_RNA,
            n_top_genes=n_top_genes,
            flavor="seurat_v3",
            subset=True
        )

    # ---- 2) ATAC peak filtering ----
    sc.pp.filter_genes(adata_ATAC, min_cells=int(adata_ATAC.shape[0] * atac_min_cell_frac))

    # ---- 3) ADT panel list ----
    adt_panel = pd.read_csv(adt_panel_path, sep="\t")
    adt_list_for_mvi = adt_panel[lineage].dropna().astype(str).tolist()

    # ---- 4) Concatenate RNA + ATAC ----
    adata_cat = ad.concat([adata_RNA, adata_ATAC], axis=1, join="outer", label=None, index_unique=None)
    adata_cat.obs = adata_RNA.obs.copy()
    adata_cat.obsm["protein_expression"] = adata_RNA.obsm["protein_expression"].loc[:, adt_list_for_mvi].copy()

    print(f"RNA HVGs kept: {adata_RNA.n_vars:,}")
    print(f"ATAC peaks kept: {adata_ATAC.n_vars:,}")
    print(f"ADT features used: {len(adt_list_for_mvi)}")
    print(f"Concatenated AnnData shape: {adata_cat.shape}")

    # ---- MultiVI ----
    scvi.model.MULTIVI.setup_anndata(
        adata_cat,
        batch_key=multivi_batch_key,
        protein_expression_obsm_key="protein_expression",
    )

    mvi = scvi.model.MULTIVI(
        adata_cat,
        n_genes=int((adata_cat.var['modality'] == 'Gene Expression').sum()),
        n_regions=int((adata_cat.var['modality'] == 'Peaks').sum()),
        n_proteins=len(adt_list_for_mvi)
    )

    mvi.view_anndata_setup()
    mvi.train(max_epochs=max_epochs, early_stopping=True)
    mvi.save(save_path)

    return mvi, adata_cat

In [ ]:
mvi, adata_cat = prepare_and_train_multivi(lineage="Global", 
                                           adt_panel_path=adt_panel_path,
                                           rna_min_cells=100, 
                                           atac_min_cell_frac=0.01,
                                           n_top_genes=3000, 
                                           batch_key_in_obs="sample",
                                           multivi_batch_key="batch_id",
                                           max_epochs=1000, 
                                           save_path="Reproducibility/Results/MultiVI/Global")

In [ ]:
mvi, adata_cat = prepare_and_train_multivi(adata = adata,
                                           lineage="CD4_T", 
                                           adt_panel_path=adt_panel_path,
                                           rna_min_cells=0, 
                                           atac_min_cell_frac=0,
                                           n_top_genes=3000, 
                                           # batch_key_in_obs="sample",
                                           multivi_batch_key="batch_id",
                                           max_epochs=1000, 
                                           save_path="Reproducibility/Results/MultiVI/CD4_T")

In [ ]:
mvi, adata_cat = prepare_and_train_multivi(adata = adata,
                                           lineage="CD8_T_NK_ILC", 
                                           adt_panel_path=adt_panel_path,
                                           rna_min_cells=0, 
                                           atac_min_cell_frac=0,
                                           n_top_genes=3000, 
                                           # batch_key_in_obs="sample",
                                           multivi_batch_key="batch_id",
                                           max_epochs=1000, 
                                           save_path="Reproducibility/Results/MultiVI/CD8_T_NK_ILC")

In [ ]:
mvi, adata_cat = prepare_and_train_multivi(adata = adata,
                                           lineage="B", 
                                           adt_panel_path=adt_panel_path,
                                           rna_min_cells=0, 
                                           atac_min_cell_frac=0,
                                           n_top_genes=3000, 
                                           # batch_key_in_obs="sample",
                                           multivi_batch_key="batch_id",
                                           max_epochs=1000, 
                                           save_path="Reproducibility/Results/MultiVI/B")

In [ ]:
mvi, adata_cat = prepare_and_train_multivi(adata = adata,
                                           lineage="Myeloid", 
                                           adt_panel_path=adt_panel_path,
                                           rna_min_cells=0, 
                                           atac_min_cell_frac=0,
                                           n_top_genes=3000, 
                                           # batch_key_in_obs="sample",
                                           multivi_batch_key="batch_id",
                                           max_epochs=1000, 
                                           save_path="Reproducibility/Results/MultiVI/Myeloid")

In [ ]:
mvi, adata_cat = prepare_and_train_multivi(adata = adata,
                                           lineage="MSC", 
                                           adt_panel_path=adt_panel_path,
                                           rna_min_cells=0, 
                                           atac_min_cell_frac=0,
                                           n_top_genes=3000, 
                                           # batch_key_in_obs="sample",
                                           multivi_batch_key="batch_id",
                                           max_epochs=1000, 
                                           save_path="Reproducibility/Results/MultiVI/MSC")

In [ ]:
mvi, adata_cat = prepare_and_train_multivi(adata = adata,
                                           lineage="Malignant", 
                                           adt_panel_path=adt_panel_path,
                                           rna_min_cells=0, 
                                           atac_min_cell_frac=0,
                                           n_top_genes=3000, 
                                           # batch_key_in_obs="sample",
                                           multivi_batch_key="batch_id",
                                           max_epochs=1000, 
                                           save_path="Reproducibility/Results/MultiVI/Malignant")

In [ ]:
adata_cat.obsm["denoised_protein"] = mvi.get_normalized_expression_wprotein(
#    n_samples=25,
    return_mean=True,
    transform_batch=adata.obs.batch_id.unique(),
)

adt_counts_denoised = pd.DataFrame(
    adata.obsm["denoised_protein"],
    index=adata.obs_names,
    columns=adata.obsm["protein_expression"].columns
)

adt_counts_denoised.to_csv("Reproducibility/Results/MultiVI/Malignant/UC_DOGMA_ADT_counts_denoised_Malignant.txt", sep='\t')

In [ ]:
def prepare_and_train_multivi_BCG(
    adata,
    rna_min_cells=0,
    atac_min_cell_frac=0,
    n_top_genes=4000,
    batch_key_in_obs="sample",
    multivi_batch_key="batch_id",
    max_epochs=1000,
    save_path="path_to_save"
):
    adata_use = adata[adata.obs["sample"].isin(["BC_011","BC_016","BC_023","BC_033","BC_037",
                              'BC_027','BC_032',"BC_039","BC_040",'BC_043',"BC_044","BC_047","BC_048"])&
                      adata.obs['lineage'].isin(['CD4_T','CD8_T_NK_ILC','B','Myeloid'])].copy()
    
    # ---- 0) Split modalities ----
    adata_RNA  = adata_use[:, adata_use.var["modality"] == "Gene Expression"].copy()
    adata_ATAC = adata_use[:, adata_use.var["modality"] == "Peaks"].copy()

    # ---- 1) RNA gene filtering ----
    if "black_list" in adata_RNA.var.columns:
        keep_mask = adata_RNA.var["black_list"].astype(str).str.lower().eq("keep")
        adata_RNA = adata_RNA[:, keep_mask].copy()

    sc.pp.filter_genes(adata_RNA, min_cells=rna_min_cells)

    sc.pp.highly_variable_genes(
        adata_RNA,
        n_top_genes=n_top_genes,
        flavor="seurat_v3",
        subset=True
    )

    # ---- 2) ATAC peak filtering ----
    sc.pp.filter_genes(adata_ATAC, min_cells=int(adata_ATAC.shape[0] * atac_min_cell_frac))

    # ---- 3) Concatenate RNA + ATAC ----
    adata_cat = ad.concat([adata_RNA, adata_ATAC], axis=1, join="outer", label=None, index_unique=None)
    adata_cat.obs = adata_RNA.obs.copy()
    adata_cat.obsm["protein_expression"] = adata_RNA.obsm["protein_expression"].copy()

    print(f"RNA HVGs kept: {adata_RNA.n_vars:,}")
    print(f"ATAC peaks kept: {adata_ATAC.n_vars:,}")
    print(f"Concatenated AnnData shape: {adata_cat.shape}")

    # ---- MultiVI ----
    scvi.model.MULTIVI.setup_anndata(
        adata_cat,
        batch_key=multivi_batch_key,
        protein_expression_obsm_key="protein_expression",
    )

    mvi = scvi.model.MULTIVI(
        adata_cat,
        n_genes=int((adata_cat.var['modality'] == 'Gene Expression').sum()),
        n_regions=int((adata_cat.var['modality'] == 'Peaks').sum()),
        n_proteins=0
    )

    mvi.view_anndata_setup()
    mvi.train(max_epochs=max_epochs, early_stopping=True)
    mvi.save(save_path)

    return mvi, adata_cat

In [ ]:
mvi, adata_cat = prepare_and_train_multivi_BCG(adata = adata, 
                                           rna_min_cells=0, 
                                           atac_min_cell_frac=0,
                                           n_top_genes=3000, 
                                           batch_key_in_obs="sample",
                                           multivi_batch_key="batch_id",
                                           max_epochs=1000, 
                                           save_path="Reproducibility/Results/MultiVI/BCG")